# Demo 05 : Compiling and APIs

Run the cells in order and watch. There is nothing to write.

## a · compile-and-visualise

We compile a builder once, inspect its Mermaid description, then invoke it.

In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class Message(TypedDict):
    text: str

builder = StateGraph(Message)
builder.add_node("normalise", lambda s: {"text": s["text"].strip().title()})
builder.add_edge(START, "normalise")
builder.add_edge("normalise", END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())
print("result:", graph.invoke({"text": "  comet snake  "}))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	normalise(normalise)
	__end__([<p>__end__</p>]):::last
	__start__ --> normalise;
	normalise --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

result: {'text': 'Comet Snake'}


`compile` checks the builder and rejects an edge to a node that does not exist, before anything runs.

In [2]:
broken = StateGraph(Message)
broken.add_node("tester", lambda s: {})
broken.add_edge(START, "tester")
broken.add_edge("tester", "publisher")
try:
    broken.compile()
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError Found edge ending at unknown node `publisher`


## b · functional-api

We express the same sequence with `entrypoint` and `task` functions.

In [3]:
from langgraph.func import entrypoint, task

@task
def normalise(text: str) -> str:
    return text.strip().title()

@entrypoint()
def workflow(text: str) -> str:
    return normalise(text).result()

print("functional result:", workflow.invoke("  comet snake  "))

functional result: Comet Snake


## Things to try

- Invoke both workflows with a different input string
- Add a second task before the entrypoint returns